# RNN Vanilla para autocompletar codigo C (paso a paso)

Este notebook explica el proyecto `3-RNN/`: una **RNN Vanilla** (
`SimpleRNN` de Keras) entrenada a nivel de **caracteres** sobre un corpus
de 150 funciones en C, capaz de sugerir continuaciones de codigo.

**Stack tecnologico**

- `tensorflow 2.16.1` para el modelo y el entrenamiento
- `numpy < 2.0` (TF 2.16 rompe con numpy 2.x)
- `fastapi` + `uvicorn` para el servidor HTTP
- `pydantic` para la validacion de requests
- `pyenv` con **Python 3.11.9** (TF 2.16 no soporta 3.14)

**Punto de entrada**

- Pipeline completo: `python src/sample_clean.py && python src/preprocess.py && python src/train.py`
- Prediccion CLI: `python src/predict.py --prompt "int sum"`
- API HTTP: `uvicorn src.api:app --port 8000`
- Servidor stdio: `echo '{...}' | python src/server_stdio.py`
- Extension VSCode: ver `vscode-extension/`

**Aclaracion importante**

Este **NO** es el editor que predice codigo C. Es la guia explicativa
para mostrarle al profesor como funciona el pipeline. La prediccion
real ocurre en la extension VSCode (F7), en `predict.py` (F4), o en el
servidor HTTP (F5).

**Convenciones**

- Las referencias al codigo se muestran como `ruta/archivo.py:linea`.
- Los snippets son copia fiel del proyecto; no se modifica ninguna linea.
- En cada paso hay un bloque **"Que le digo al profesor"** con un
  guion corto que se puede leer literalmente.


## 1. Arquitectura y pipeline F1-F7

La actividad exige 7 entregables, cada uno nombrado con una letra F:

```
F1: sample_clean.py      Corpus crudo (466 MB) -> 150 funciones limpias (~70 KB)
                              |
                              v
F2: preprocess.py        Texto -> ventanas char-level (X.npy, Y.npy)
                              |
                              v
F3: train.py             Define + entrena la RNN, guarda rnn_v1.keras
                              |
                              v
F4: predict.py           complete() + complete_full(), CLI
                              |
                +-------------+-------------+
                |                           |
                v                           v
F5: api.py (HTTP)                  F6: server_stdio.py (JSON-line)
   POST /predict, /suggest             stdin/stdout para el editor
                                              |
                                              v
                                      F7: vscode-extension/
                                          extension.js + package.json
                                          Ctrl+Shift+Space en .c
```

**Diagrama de los datos en disco**

```
dataset/
  funciones.c                      <- 466 MB, corpus crudo (F1 lo lee)
  sample/
    funciones.c                    <- ~70 KB, 150 funciones limpias (F1)
    REPORTE.md                     <- metricas de muestreo (F1)
  processed/
    X.npy                          <- (N, 32) int64, ventanas de entrada (F2)
    Y.npy                          <- (N, 32) int64, ventanas target (F2)
    meta.json                      <- vocab, block_size, params (F2)
models/
  rnn_v1.keras                     <- modelo entrenado (F3)
  rnn_v1.meta.json                 <- copiado de processed + params (F3)
  rnn_v1.history.json              <- curva de perdida (F3)
notebooks/
  entrenamiento.ipynb              <- entregable formal de la actividad
  explicacion_rnn_autocomplete_paso_a_paso.ipynb  <- ESTE notebook
```


## 2. Estructura de carpetas (mapa)

```
3-RNN/
|-- README.md                      # guia principal
|-- (sin .venv; el venv vive en la raiz: ~/IA/.venv)
|-- (sin .python-version; usa el de la raiz)
|-- GUIA_DEMO.md                   # 4 modalidades de demo al profesor
|-- GUIA_EXPLICACION.md            # guion para explicar el proyecto
|-- requirements.txt               # tensorflow, fastapi, etc.
|-- dataset/                       # datos de entrenamiento
|-- models/                        # modelos entrenados
|-- notebooks/                     # este notebook + el entregable formal
|-- src/                           # codigo fuente
|   |-- sample_clean.py            # F1
|   |-- preprocess.py              # F2
|   |-- train.py                   # F3
|   |-- predict.py                 # F4
|   |-- api.py                     # F5
|   '-- server_stdio.py            # F6
|-- vscode-extension/              # F7
|   |-- package.json
|   |-- extension.js
|   '-- test_e2e.js
'-- docs/                          # documentacion escrita (md)
```

**Mapa rapido de archivos clave**

| Archivo | Una linea de responsabilidad |
|---|---|
| `src/sample_clean.py` | F1: heuristicas + extraccion de funciones |
| `src/preprocess.py` | F2: vocabulario + ventanas char-level |
| `src/train.py` | F3: arquitectura + fit + save |
| `src/predict.py` | F4: `RNNModel` + `complete()` + `complete_full()` |
| `src/api.py` | F5: FastAPI con `/predict` y `/suggest` |
| `src/server_stdio.py` | F6: JSON-line por stdin/stdout |
| `vscode-extension/extension.js` | F7: cliente Node que spawn-ea F6 |
| `vscode-extension/package.json` | F7: comandos, keybindings, settings |
| `models/rnn_v1.keras` | el modelo entrenado (17 854 params) |
| `models/rnn_v1.meta.json` | vocab + block_size + params |
| `models/rnn_v1.history.json` | curva de perdida por epoca |

**Que le digo al profesor**

> *El proyecto esta dividido en 7 entregables secuenciales: F1 prepara
> el corpus, F2 lo vectoriza, F3 entrena, F4 expone `complete()`. A
> partir de ahi hay dos formas de servir el modelo: F5 (HTTP con FastAPI)
> para pruebas remotas y F6 (JSON-line por stdin/stdout) que es lo que
> usa la extension de VSCode en F7.*


## 3. Paso 1 (F1): muestreo del corpus

`src/sample_clean.py` toma los 466 MB de `dataset/funciones.c` y extrae
150 funciones que cumplen heuristicas de estilo consistentes.

**Heuristicas** (`is_clean_function()`)

- Longitud entre **30 y 1500 chars**.
- **Sin** `int main(...)` (queremos librerias, no entry points).
- **ASCII puro**: rechaza funciones con comentarios en otros idiomas.
- **Llaves balanceadas**: `count('{') == count('}')`.
- **Firma reconocible**: regex `SIG_RE` que matchea `static inline int foo(...)` etc.
- **Nombre no reservado**: rechaza funciones llamadas `if`, `for`, `while`, etc.

**Regex de firma**

```python
SIG_RE = re.compile(
    r"^\s*(?:static\s+)?(?:inline\s+)?"
    r"(?:int|void|char|float|double|long|short|unsigned|signed|size_t|ssize_t|bool|"
    r"int8_t|int16_t|int32_t|int64_t|uint8_t|uint16_t|uint32_t|uint64_t)\s*\*?\s+"
    r"([A-Za-z_]\w*)\s*\([^;]*\)\s*\{",
    re.MULTILINE,
)
```

**Extraccion por conteo de profundidad**

Para cada firma encontrada, `extract_functions()` cuenta llaves con un
contador `depth` desde la `{` hasta llegar a `depth == 0`. Asi encuentra
el bloque completo aunque tenga llaves anidadas.

**Score de seleccion**

```python
def score(item):
    s = count_style(item[1])
    return s['returns'] * 2 + s['ifs'] + s['loops'] - abs(200 - len(item[1])) // 50
```

Prioriza funciones con `return`, `if` y `for/while`, y penaliza
funciones muy cortas o muy largas. Se queda con las 150 mejores.

**Salida**

- `dataset/sample/funciones.c`: las 150 funciones concatenadas (~70 KB).
- `dataset/sample/REPORTE.md`: metricas del muestreo + 5 funciones
  aleatorias como muestra.

**Que le digo al profesor**

> *Tengo un corpus crudo de 466 MB con miles de funciones. Aplico
> heuristicas de estilo: ASCII puro, sin `main()`, llaves balanceadas,
> firma valida, longitud razonable. De ahi saco 150 funciones que
> representan codigo C idiomático y consistente, ~70 KB en total.*


## 4. Paso 2 (F2): preprocesamiento char-level

`src/preprocess.py` toma el archivo `funciones.c` y lo convierte en
ventanas deslizantes de 32 caracteres, listas para entrenar.

**Pipeline**

```
dataset/sample/funciones.c  (texto)
           |
           v
chars = sorted(set(text))    # 94 chars unicos (letras, digitos, simbolos C)
           |
           v
stoi = {ch: i}                # char -> int
itos = {i: ch}                # int -> char
           |
           v
seq = [stoi[c] for c in text]  # secuencia de int64
           |
           v
for i in range(len(seq) - 32):
    X[i] = seq[i  : i+32 ]     # ventana actual
    Y[i] = seq[i+1: i+33]      # ventana desplazada en 1
           |
           v
save X.npy, Y.npy, meta.json
```

**Codigo**

```python
text = args.input.read_text(encoding='utf-8', errors='ignore')
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}

seq = np.array([stoi[c] for c in text], dtype=np.int64)
vocab_size = len(chars)

n_windows = len(seq) - args.block_size
X = np.empty((n_windows, args.block_size), dtype=np.int64)
Y = np.empty((n_windows, args.block_size), dtype=np.int64)
for i in range(n_windows):
    X[i] = seq[i : i + args.block_size]
    Y[i] = seq[i + 1 : i + 1 + args.block_size]

np.save(args.output_dir / 'X.npy', X)
np.save(args.output_dir / 'Y.npy', Y)
```

**Diagrama de ventana**

```
texto:    i  n  t     s  u  m  (  i  n  t     a  ,
pos:      0  1  2  3  4  5  6  7  8  9 10 11 12 13
seq:     [7, 3, 8, ..., ids]

X[0]:     seq[0:32]   =  [7, 3, 8, ..., seq[31]]     (32 chars)
Y[0]:     seq[1:33]   =  [3, 8, ..., seq[32]]         (32 chars, shifted by 1)

Significado: dado X[i], el modelo tiene que predecir Y[i] char a char.
```

**Salida**

| Archivo | Contenido |
|---|---|
| `X.npy` | `int64` shape `(N, 32)`, ventanas de entrada |
| `Y.npy` | `int64` shape `(N, 32)`, ventanas target (desplazadas en 1) |
| `meta.json` | `chars`, `block_size=32`, `vocab_size=94`, `corpus_chars`, `num_windows` |

**Que le digo al profesor**

> *Trabajo a nivel de caracter, no de token. El vocabulario son los 94
> caracteres unicos que aparecen en el corpus. Cada ventana es de 32
> chars: la entrada X y el target Y son iguales pero desplazadas en 1
> posicion. Asi, en cada paso temporal, el modelo aprende a predecir
> el siguiente caracter dado el contexto.*


## 5. Paso 3 (F3): arquitectura del modelo

`src/train.py:build_model()` define la red. Es deliberadamente pequena y
cumple el requisito literal de la actividad: **RNN Vanilla** (`SimpleRNN`),
sin LSTM ni GRU.

**Codigo**

```python
def build_model(vocab_size, embed_dim=48, hidden=64):
    return Sequential([
        Input(shape=(None,)),
        Embedding(vocab_size, embed_dim),
        SimpleRNN(hidden, activation='tanh', return_sequences=True),
        TimeDistributed(Dense(vocab_size)),
    ])
```

**Diagrama de capas**

```
Input(shape=(BLOCK_SIZE,))                  # 32 ints
   |
v
Embedding(vocab_size=94, embed_dim=48)     # (32, 48)
   |
v
SimpleRNN(hidden=64, tanh, return_sequences=True)   # (32, 64)
   |
v
TimeDistributed(Dense(vocab_size=94))      # (32, 94)  <- logits del sig. char
```

**Por que `SimpleRNN` y no LSTM/GRU?**

La actividad exige **RNN Vanilla** explicitamente. La diferencia:

- `SimpleRNN`: `h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)`. Sin
  compuertas. Memoria limitada al `BLOCK_SIZE` (32 chars).
- `LSTM` / `GRU`: tienen compuertas (input/forget/output) que les
  permiten mantener dependencias largas. **No se usan aca** por
  requisito de la pauta.

**Por que `return_sequences=True` + `TimeDistributed(Dense)`?**

Necesitamos un logit en **cada paso temporal** para comparar con el
caracter **siguiente** en esa misma posicion (recordemos que `Y[i] =
seq[i+1:i+33]`). Sin `return_sequences=True` solo tendriamos el ultimo
estado y dariamos un solo logit. Con `TimeDistributed` aplicamos la
misma `Dense` en cada paso, dando shape `(32, vocab_size)`.

**Por que `Embedding` y no one-hot?**

Con `vocab_size=94`, one-hot daria un vector de 94 dims. La `Embedding`
lo comprime a 48 dims aprendibles, lo que es mas eficiente y ademas
aprende relaciones entre caracteres (los digitos quedan cerca de los
digitos, etc.).

**Hiperparametros por defecto**

| Parametro | Valor | Justificacion |
|---|---|---|
| `vocab_size` | 94 | tamano del vocabulario del corpus |
| `embed_dim` | 48 | compacto, ~94*48 = 4 512 params |
| `hidden` | 64 | balance entre capacidad y velocidad CPU |
| `activation` | tanh | requerido por SimpleRNN |
| `optimizer` | Adam(1e-3) | standard para RNNs pequenas |
| `loss` | SparseCategoricalCrossentropy(from_logits=True) | labels son ints, salida son logits |
| `epochs` | 80 | curva convergente en este dataset |
| `batch_size` | 128 | standard |
| params totales | **17 854** | ver `rnn_v1.meta.json` |

**Que le digo al profesor**

> *La red es chiquita a proposito: 17 854 parametros, una sola capa
> `SimpleRNN` con 64 unidades, embedding de 48 dimensiones. Cumple el
> requisito literal de la actividad: RNN Vanilla, no LSTM ni GRU. La
> salida es un Dense aplicado a cada paso temporal, asi obtenemos un
> logit por cada posicion y predecimos el siguiente caracter.*


## 6. Paso 4 (F3): entrenamiento

`src/train.py:main()` carga los datos preprocesados, compila el modelo
y entrena 80 epocas con seed fija para reproducibilidad.

**Codigo clave**

```python
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

model.compile(
    optimizer=Adam(learning_rate=args.lr),
    loss=SparseCategoricalCrossentropy(from_logits=True),
)

history = model.fit(
    X, Y,
    epochs=args.epochs,        # 80
    batch_size=args.batch_size,  # 128
    verbose=0,
    callbacks=[PrintEvery(every=10)],
)
```

**Callback `PrintEvery`**

```python
class PrintEvery(Callback):
    def __init__(self, every=10):
        super().__init__()
        self.every = every

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every == 0 or epoch == 0:
            logs = logs or {}
            print(f"  epoch {epoch + 1:4d} | loss={logs.get('loss', 0):.4f}")
```

Imprime la perdida cada 10 epocas (y la primera) sin el spam de
`verbose=1`. Asi la salida queda limpia pero uno ve el progreso.

**Sub-muestreo para velocidad**

```python
if len(X) > args.max_windows:
    idx = np.random.RandomState(SEED).permutation(len(X))[: args.max_windows]
    idx.sort()
    X, Y = X[idx], Y[idx]
```

Con `--max-windows 20000` (default), se submuestrea de ~70 000 a 20 000
ventanas para que el entrenamiento corra en ~5 min en CPU. Con la
misma seed el submuestreo es reproducible.

**Artefactos generados**

- `models/rnn_v1.keras`: el modelo (244 KB, ~17 854 params).
- `models/rnn_v1.meta.json`: copia de meta + `model_path` + `params`.
- `models/rnn_v1.history.json`: lista de `loss` por epoca + `initial_loss` + `final_loss`.

**Numeros del entrenamiento real** (reproducibles con seed=42)

| Metrica | Valor |
|---|---|
| Vocabulario | 94 chars unicos |
| Corpus | 69 874 chars (~70 KB) |
| Ventanas | 69 842 (sub-muestreadas a 20 000) |
| Epochs | 80 |
| Perdida inicial | 3.07 |
| Perdida final | 1.06 |
| Tiempo (CPU, 4 cores) | ~5 min |
| Parametros | 17 854 |

La entropia inicial `ln(94) ~ 4.54` representa el peor caso (prediccion
uniforme). Una perdida de 1.06 significa que el modelo es **~4x mas
preciso que azar** a la hora de elegir el siguiente caracter.

**Que le digo al profesor**

> *Entreno 80 epocas con Adam(1e-3) y cross-entropy sobre labels enteros.
> La perdida arranca en 3.07 (cerca del azar de ln(94)=4.54) y baja a
> 1.06, o sea ~4x mejor que azar. En CPU tarda 5 minutos con 20 000
> ventanas. Los pesos se guardan en formato .keras.*


## 7. Paso 5 (F4): prediccion autorregresiva

`src/predict.py` carga el modelo y la meta, y expone la funcion
`complete()` que genera caracteres uno a uno.

**Carga del modelo**

```python
@dataclass
class RNNModel:
    model: tf.keras.Model
    stoi: dict[str, int]
    itos: dict[int, str]
    block_size: int
    vocab_size: int

    @classmethod
    def load(cls, model_path, meta_path=None):
        if meta_path is None:
            meta_path = model_path.with_suffix('.meta.json')
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        chars = meta['chars']
        stoi = {ch: i for i, ch in enumerate(chars)}
        itos = {i: ch for ch, i in stoi.items()}
        model = tf.keras.models.load_model(model_path)
        return cls(model=model, stoi=stoi, itos=itos,
                   block_size=meta['block_size'],
                   vocab_size=meta['vocab_size'])
```

**Encoding del prefijo**

```python
def _encode(prefix, stoi, block_size):
    ids = [stoi[c] for c in prefix if c in stoi]
    if len(ids) >= block_size:
        ids = ids[-block_size:]   # truncar al final
    else:
        pad_id = 0
        ids = [pad_id] * (block_size - len(ids)) + ids  # pad al inicio
    return np.array(ids, dtype=np.int64).reshape(1, block_size)
```

**Sampling con temperatura**

```python
def _sample(logits, temperature):
    if temperature <= 0.0:
        return int(np.argmax(logits))     # greedy
    logits = logits / temperature
    logits = logits - logits.max()       # softmax estable
    probs = np.exp(logits)
    probs = probs / probs.sum()
    return int(np.random.choice(len(probs), p=probs))
```

- `temperature=0`: siempre `argmax` (greedy, determinista).
- `temperature>0`: cuanto mas alta, mas uniform -> mas variedad. Mas
  baja, mas picos -> mas determinista. Default `0.4`.

**Loop autorregresivo**

```python
def complete(rnn, prefix, max_new=60, temperature=0.5, seed=None):
    if seed is not None:
        np.random.seed(seed)

    state = _encode(prefix, rnn.stoi, rnn.block_size)  # (1, 32)
    generated = []
    for _ in range(max_new):
        logits = rnn.model.predict(state, verbose=0)[0, -1]  # (vocab,)
        idx = _sample(logits, temperature)
        generated.append(rnn.itos[idx])
        # slide window: descartamos el primer char y agregamos idx al final
        state = np.concatenate(
            [state[:, 1:], np.array([[idx]], dtype=np.int64)],
            axis=1,
        )
    return ''.join(generated)
```

**Diagrama del loop**

```
state = [c1, c2, ..., c32]                  (32 chars del prefijo)
for _ in range(max_new):
    logits = model.predict(state)[0, -1]    (vector de tamanho vocab)
    idx    = sample(logits, T)
    output.append(idx)
    state  = [c2, ..., c32, idx]             (slide window)
return ''.join(output)
```

**CLI**

```
$ python src/predict.py --prompt "int sum" --max-new 60 --temperature 0.4
int sum (int a, int b) { return a + b; }
```

**Que le digo al profesor**

> *La prediccion es autorregresiva: tomo el prefijo, lo encodeo a una
> ventana de 32 ids, le pido al modelo la distribucion sobre el siguiente
> caracter, sampleo uno con temperatura, lo concateno al final de la
> ventana y repito. Con temperatura 0.4 el modelo es ~70% determinista
> manteniendo algo de variedad.*


## 8. Paso 6 (F5): API HTTP con FastAPI

`src/api.py` expone el modelo como servicio HTTP. Es la forma mas
simple de consumirlo desde un cliente externo (Postman, curl, otro
servicio, etc.).

**Codigo**

```python
app = FastAPI(title='RNN Autocomplete API', version='0.1.0')
_rnn: RNNModel | None = None

def get_rnn():
    global _rnn
    if _rnn is None:
        _rnn = RNNModel.load(DEFAULT_MODEL)
    return _rnn

class PredictRequest(BaseModel):
    prompt: str
    max_new: int = Field(default=60, ge=1, le=400)
    temperature: float = Field(default=0.5, ge=0.0, le=2.0)

class SuggestRequest(BaseModel):
    prefix: str
    n: int = Field(default=5, ge=1, le=20)

@app.get('/health')
def health(): return {'status': 'ok'}

@app.post('/predict')
def predict(req: PredictRequest):
    rnn = get_rnn()
    cont = complete(rnn, req.prompt, req.max_new, req.temperature)
    return {'completion': cont, 'full': req.prompt + cont}

@app.post('/suggest')
def suggest(req: SuggestRequest):
    rnn = get_rnn()
    state = _encode(req.prefix, rnn.stoi, rnn.block_size)
    logits = rnn.model.predict(state, verbose=0)[0, -1]
    top = np.argsort(-logits)[: req.n]
    return {'items': [rnn.itos[int(i)] for i in top]}
```

**Endpoints**

| Metodo | Ruta | Body | Respuesta |
|---|---|---|---|
| `GET` | `/health` | - | `{status: ok}` |
| `POST` | `/predict` | `{prompt, max_new?, temperature?}` | `{completion, full}` |
| `POST` | `/suggest` | `{prefix, n?}` | `{items: [str x N]}` (top-N greedy) |

**Validacion con Pydantic**

`Field(ge=1, le=400)` en `max_new` y `Field(ge=0.0, le=2.0)` en
`temperature` validan automaticamente los rangos. Si un cliente manda
`max_new=1000`, FastAPI responde 422 Unprocessable Entity con el detalle.

**Lazy load del modelo**

`get_rnn()` carga el modelo en memoria solo la primera vez que se
llama a `/predict` o `/suggest`. Asi el server arranca rapido y solo
paga el costo del modelo cuando recibe la primera request.

**Ejemplo con curl**

```
$ uvicorn src.api:app --port 8000 &
$ curl -X POST http://127.0.0.1:8000/predict \\
       -H 'Content-Type: application/json' \\
       -d '{"prompt":"int sum","max_new":40,"temperature":0.4}'
{"completion":"(int a, int b) { return a + b; }",
 "full":"int sum(int a, int b) { return a + b; }"}
```

**Diagrama**

```
cliente --POST /predict--> FastAPI --get_rnn()--> _rnn (singleton)
                                     |
                                     v
                              complete()
                                     |
                                     v
                              model.predict()
                                     |
                                     v
cliente <--{completion, full}-- logits -> sample
```

**Que le digo al profesor**

> *La API HTTP es para integraciones externas o pruebas rapidas con
> curl. Tiene tres endpoints: `/health` para liveness, `/predict`
> para generar una continuacion, y `/suggest` para ver los top-N
> siguientes caracteres sin sampling. La validacion con Pydantic
> rechaza automaticamente parametros fuera de rango.*


## 9. Paso 7 (F6): servidor stdio JSON-line

`src/server_stdio.py` es el mismo modelo expuesto por **stdin/stdout**
en formato JSON-line. Es la forma que usa la extension VSCode (F7).

**Protocolo**

```
Request  : {id, method, ...args}
Response : {id, ok, ...result}

Cada linea es un JSON independiente. Un flush por linea.
```

**Codigo**

```python
def main():
    rnn = _load_rnn()
    for raw in sys.stdin:
        line = raw.strip()
        if not line: continue
        try:
            req = json.loads(line)
            req_id = req.get('id')
            method = req.get('method')
            if method == 'complete':
                payload = handle_complete(rnn, req)
            elif method == 'suggest':
                payload = handle_suggest(rnn, req)
            elif method == 'ping':
                payload = {'pong': True}
            else:
                raise ValueError(f'unknown method: {method!r}')
            resp = {'id': req_id, 'ok': True, **payload}
        except Exception as exc:
            resp = {
                'id': req.get('id') if isinstance(req, dict) else None,
                'ok': False, 'error': str(exc),
                'trace': traceback.format_exc(limit=2),
            }
        sys.stdout.write(json.dumps(resp, ensure_ascii=False) + '\n')
        sys.stdout.flush()
```

**Metodos**

| Metodo | Args | Respuesta |
|---|---|---|
| `complete` | `prefix, max_new?, temperature?` | `text` (continuacion) |
| `suggest` | `prefix, n?` | `items` (top-N chars) |
| `ping` | - | `pong: true` |

**Ejemplo**

```
$ echo '{"id":1,"method":"complete","prefix":"int sum","max_new":30}' \\
       | python src/server_stdio.py
{"id": 1, "ok": true, "text": "(int a, int b) { return a + b; }"}
```

**Por que este formato?**

VSCode (y cualquier editor) puede hacer `spawn` del script y leer/escribir
lineas. No necesita HTTP, ni sockets, ni NamedPipes: solo stdin/stdout
estan disponibles siempre. JSON-line es trivial de parsear linea por
linea y soporta concurrencia trivial con `id` por request.

**Por que NO es un thread pool?**

Las predicciones son **secuenciales** dentro de un mismo proceso: cada
`predict()` toma ~5-10 ms en CPU, asi que atender una request a la vez
es suficiente. Si quisieras concurrencia real, habria que levantar
varios procesos (gunicorn/uvicorn workers para F5; varios `spawn`s
para F6).

**Que le digo al profesor**

> *El servidor stdio es lo que usa la extension de VSCode. Lee una linea
> JSON por stdin, dispatcha segun el metodo (`complete`, `suggest`,
> `ping`), escribe una linea JSON con la respuesta, y repite. Es simple,
> no necesita HTTP ni sockets, y cualquier editor puede hacer spawn.*


## 10. Paso 8 (F7): extension de VSCode

`vscode-extension/extension.js` es un cliente Node que spawn-ea el
servidor stdio y le manda requests cuando el usuario aprieta
**Ctrl+Shift+Space** en un archivo `.c`.

**Comandos y keybindings** (`package.json`)

```json
{
  "commands": [
    {"command": "rnnC.complete", "title": "RNN: completar linea", "category": "RNN"},
    {"command": "rnnC.suggest",  "title": "RNN: elegir siguiente caracter", "category": "RNN"},
    {"command": "rnnC.status",   "title": "RNN: estado del servidor", "category": "RNN"}
  ],
  "keybindings": [
    {"command": "rnnC.complete", "key": "ctrl+shift+space",
     "when": "editorTextFocus && editorLangId == c"}
  ],
  "configuration": {
    "rnnC.pythonPath":   {"type": "string",  "default": "python"},
    "rnnC.serverScript": {"type": "string",  "default": "src/server_stdio.py"},
    "rnnC.modelPath":    {"type": "string",  "default": "models/rnn_v1.keras"},
    "rnnC.maxNew":       {"type": "integer", "default": 60},
    "rnnC.temperature":  {"type": "number",  "default": 0.4}
  }
}
```

**Flujo de un `complete`** (extension.js)

```javascript
async function cmdComplete() {
  const editor = vscode.window.activeTextEditor;
  if (!editor) return;
  const cfg = vscode.workspace.getConfiguration('rnnC');
  const prefix = linePrefix(editor);    // texto desde inicio de linea hasta cursor
  try {
    const r = await request('complete', {
      prefix,
      max_new: cfg.get('maxNew', 60),
      temperature: cfg.get('temperature', 0.4),
    });
    await editor.edit((b) => b.insert(editor.selection.active, r.text));
  } catch (err) {
    vscode.window.showErrorMessage(`RNN complete failed: ${err.message}`);
  }
}

function linePrefix(editor) {
  const pos = editor.selection.active;
  return editor.document.getText(
    new vscode.Range(new vscode.Position(pos.line, 0), pos)
  );
}
```

**Manejo de stdin/stdout con buffer**

El server manda varias lineas; `extension.js` mantiene un `buffer`
global y va cortando por `\n`. Cada linea se parsea como JSON, se busca
el callback con el `id` correspondiente y se resuelve la Promise.

```javascript
serverProc.stdout.on('data', (chunk) => {
    buffer += chunk.toString('utf8');
    let nl;
    while ((nl = buffer.indexOf('\n')) !== -1) {
        const line = buffer.slice(0, nl).trim();
        buffer = buffer.slice(nl + 1);
        if (!line) continue;
        const msg = JSON.parse(line);
        const cb = pending.get(msg.id);
        if (cb) { pending.delete(msg.id); cb(msg); }
    }
});
```

**Diagrama end-to-end**

```
VSCode (extension.js)
    |
    |  spawn(python, [src/server_stdio.py, models/rnn_v1.keras])
    v
python src/server_stdio.py
    |
    |  carga rnn_v1.keras + meta.json  (una vez al arrancar)
    |  lee lineas JSON de stdin
    |
    |  --{id:1, method:complete, prefix:"int sum"}-->
    |
    |  <--{id:1, ok:true, text:"(int a, int b) { return a + b; }"}--
    |
    v
VSCode inserta el texto en el cursor
```

**Como se instala**

- **Modo desarrollo (F5)**:
  ```
  code vscode-extension/
  # dentro de VSCode, F5
  ```
  Abre una Extension Development Host con la extension cargada.

- **Instalacion permanente**:
  ```
  cd vscode-extension
  npm install -g @vscode/vsce
  vsce package
  code --install-extension rnn-c-autocomplete-0.1.0.vsix
  ```

**Que le digo al profesor**

> *La extension es un cliente Node chiquito. Hace spawn del servidor
> Python, manda una linea JSON con el prefijo cuando el usuario aprieta
> Ctrl+Shift+Space, recibe la respuesta y la inserta en el cursor. El
> servidor queda vivo entre requests, asi las siguientes son casi
> instantaneas.*


## 11. Celda ejecutable 1: inspeccion de `meta.json`

La siguiente celda **NO requiere TensorFlow**. Solo usa el modulo
`json` para leer los artefactos del modelo ya entrenado y `matplotlib`
para graficar la distribucion del vocabulario.

Output esperado:
- Tabla con `vocab_size`, `block_size`, `corpus_chars`, `num_windows`,
  `params`.
- Histograma de los 94 caracteres del vocabulario (no se ve interesante
  per se, pero sirve para ver que el corpus es ASCII puro).


In [ ]:
import json
from pathlib import Path
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "3-RNN":
    PROJECT_ROOT = PROJECT_ROOT / "3-RNN"
meta = json.loads((PROJECT_ROOT / 'models/rnn_v1.meta.json').read_text(encoding='utf-8'))
history = json.loads((PROJECT_ROOT / 'models/rnn_v1.history.json').read_text(encoding='utf-8'))

print('=== rnn_v1.meta.json ===')
for key in ['vocab_size', 'block_size', 'corpus_chars', 'num_windows']:
    print(f'  {key:<14s} = {meta[key]}')
print('  model_path     =', meta.get('model_path', '(no guardado)'))
print('  params         =', json.dumps(meta.get('params', {}), indent=2))
print()
print('Vocabulario (94 chars):')
chars = meta['chars']
print(' ', ''.join(chars))
print()
print('=== rnn_v1.history.json ===')
print(f'  initial_loss   = {history["initial_loss"]:.4f}')
print(f'  final_loss     = {history["final_loss"]:.4f}')
print(f'  total_epochs   = {len(history["loss"])}')
print(f'  ln(vocab_size) = {__import__("math").log(meta["vocab_size"]):.4f}  (azar uniforme)')
print(f'  mejora vs azar = {__import__("math").log(meta["vocab_size"]) / history["final_loss"]:.2f}x')

sample = (PROJECT_ROOT / 'dataset/sample/funciones.c').read_text(encoding='utf-8')
freq = Counter(sample)
freq_in_vocab = [(c, freq.get(c, 0)) for c in chars]
freq_in_vocab.sort(key=lambda x: -x[1])
fig, ax = plt.subplots(figsize=(10, 4))
labels = [c if c != ' ' else 'SP' for c, _ in freq_in_vocab[:30]]
counts = [n for _, n in freq_in_vocab[:30]]
ax.bar(range(len(counts)), counts)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(labels, rotation=0, fontsize=8)
ax.set_title('Top 30 chars del corpus (mas frecuentes primero)')
ax.set_ylabel('conteo')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
out = Path('/tmp/opencode/rnn_vocab_freq.png')
fig.savefig(out, dpi=120)
plt.close(fig)
print()
print(f'Histograma guardado en {out}')


## 12. Celda ejecutable 2: curva de perdida

Grafica la perdida de las 80 epocas. La linea horizontal punteada
marca `ln(94) ~ 4.54` (azar uniforme: predecir el caracter correcto
tiene 1/94 de probabilidad, entropia = `ln(94)`).

Output: archivo `/tmp/opencode/rnn_loss_curve.png`.


In [ ]:
import json
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "3-RNN":
    PROJECT_ROOT = PROJECT_ROOT / "3-RNN"
history = json.loads(
    (PROJECT_ROOT / 'models/rnn_v1.history.json').read_text(encoding='utf-8')
)
meta = json.loads(
    (PROJECT_ROOT / 'models/rnn_v1.meta.json').read_text(encoding='utf-8')
)

loss = history['loss']
epochs = list(range(1, len(loss) + 1))
uniform = math.log(meta['vocab_size'])  # azar uniforme

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, loss, color='steelblue', linewidth=2, label='loss (entrenamiento)')
ax.axhline(uniform, color='red', linestyle='--', alpha=0.7,
           label=f'azar uniforme: ln({meta["vocab_size"]}) = {uniform:.2f}')
ax.axhline(loss[0], color='gray', linestyle=':', alpha=0.5,
           label=f'perdida inicial = {loss[0]:.2f}')
ax.axhline(loss[-1], color='green', linestyle=':', alpha=0.5,
           label=f'perdida final = {loss[-1]:.2f}')
ax.fill_between(epochs, loss[-1], uniform, alpha=0.1, color='green')
ax.set_xlabel('epoca')
ax.set_ylabel('SparseCategoricalCrossentropy')
ax.set_title('RNN Vanilla char-level C - curva de perdida (80 epocas, 20K ventanas)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, uniform + 0.5)
plt.tight_layout()
out = Path('/tmp/opencode/rnn_loss_curve.png')
fig.savefig(out, dpi=120)
plt.close(fig)
print(f'Curva guardada en {out}')
print(f'  initial_loss = {loss[0]:.4f}  (~ {(loss[0]/uniform)*100:.0f}% del azar)')
print(f'  final_loss   = {loss[-1]:.4f}  (~ {(loss[-1]/uniform)*100:.0f}% del azar)')
print(f'  mejora       = {uniform/loss[-1]:.2f}x mejor que azar')


## 13. Celda ejecutable 3: inspeccion del corpus limpio

Lee `dataset/sample/funciones.c` y muestra:
- Cantidad total de caracteres y lineas.
- Cantidad de funciones (regex `SIG_RE`).
- Tres funciones de muestra (primera, mitad, ultima).
- Histograma de longitudes.

Output: archivo `/tmp/opencode/rnn_func_lengths.png` con el histograma.


In [ ]:
import re
from pathlib import Path
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "3-RNN":
    PROJECT_ROOT = PROJECT_ROOT / "3-RNN"
path = PROJECT_ROOT / 'dataset/sample/funciones.c'
text = path.read_text(encoding='utf-8', errors='ignore')

SIG_RE = re.compile(
    r'^\s*(?:static\s+)?(?:inline\s+)?'
    r'(?:int|void|char|float|double|long|short|unsigned|signed|size_t|ssize_t|bool|'
    r'int8_t|int16_t|int32_t|int64_t|uint8_t|uint16_t|uint32_t|uint64_t)\s*\*?\s+'
    r'([A-Za-z_]\w*)\s*\([^;]*\)\s*\{',
    re.MULTILINE,
)

def extract(text, sig_re):
    matches = list(sig_re.finditer(text))
    blocks = []
    for i, m in enumerate(matches):
        start = m.start()
        cap = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        depth = 0
        seen = False
        end = cap
        for j in range(start, cap):
            c = text[j]
            if c == '{':
                depth += 1; seen = True
            elif c == '}' and seen:
                depth -= 1
                if depth == 0:
                    end = j + 1
                    break
        blocks.append(text[start:end])
    return blocks

blocks = extract(text, SIG_RE)
lengths = [len(b) for b in blocks]

print(f'Archivo: {path}')
print(f'  bytes:       {path.stat().st_size:,}')
print(f'  chars:       {len(text):,}')
print(f'  lineas:      {text.count(chr(10)):,}')
print(f'  funciones:   {len(blocks)}')
print(f'  longitud min: {min(lengths)}')
print(f'  longitud max: {max(lengths)}')
print(f'  longitud prom: {sum(lengths)/len(lengths):.1f}')
print()
print('=== Muestra: primera funcion ===')
print(blocks[0][:400] + ('...' if len(blocks[0]) > 400 else ''))
print()
print('=== Muestra: funcion del medio ===')
mid = blocks[len(blocks) // 2]
print(mid[:400] + ('...' if len(mid) > 400 else ''))
print()
print('=== Muestra: ultima funcion ===')
print(blocks[-1][:400] + ('...' if len(blocks[-1]) > 400 else ''))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('longitud de la funcion (chars)')
ax.set_ylabel('cantidad')
ax.set_title(f'Distribucion de longitudes ({len(blocks)} funciones)')
ax.axvline(sum(lengths)/len(lengths), color='red', linestyle='--',
           label=f'promedio = {sum(lengths)/len(lengths):.0f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = Path('/tmp/opencode/rnn_func_lengths.png')
fig.savefig(out, dpi=120)
plt.close(fig)
print()
print(f'Histograma guardado en {out}')


## 14. Preguntas tipo examen (con respuestas cortas)

Estas son las preguntas que el profesor suele hacer, con la respuesta
copiada casi literal del README. Sirven para defender el proyecto.

**1. Por que `SimpleRNN` y no LSTM/GRU?**
La actividad 3 exige RNN Vanilla explicitamente. La diferencia es que
`SimpleRNN` no tiene compuertas: `h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)`.
Esto limita la memoria a ~32 chars (nuestra `BLOCK_SIZE`).

**2. Por que char-level?**
Vocab de 94 (< 100) cabe en una `Embedding` pequena; entrena rapido en
CPU y no requiere tokenizer externo.

**3. Por que `return_sequences=True`?**
Necesitamos un logit en **cada** paso para comparar con el caracter
**siguiente** en cada posicion. Sin esto solo tendriamos el ultimo
estado. Con `return_sequences=True` + `TimeDistributed(Dense)`
obtenemos `(batch, 32, vocab_size)`.

**4. Como se conecta con el editor?**
VSCode hace `spawn` de `python -m src.server_stdio` y envia lineas
JSON. El servidor mantiene el modelo en memoria y responde. La
extension (`extension.js`) hace el manejo de stdin/stdout con un
buffer y mapea por `id`.

**5. Por que no LM Studio / Ollama?**
Esas herramientas son del Proyecto 4 (Fine-Tuning / RAG con LLMs).
Proyecto 3 es RNN Vanilla. La pauta lo pide literalmente.

**6. Por que solo 150 funciones?**
La actividad pide "por lo menos 60". 150 da variedad sin saturar la
RNN; mas datos + modelo mas pequeno = mismas epocas necesarias.

**7. Como mide la calidad?**
Loss por caracter (cross-entropy). La calidad "humana" se ve generando
ejemplos: el modelo aprende sintaxis basica de C pero no es ejecutable.
La entropia inicial `ln(94) ~ 4.54` es el peor caso (azar uniforme).
Bajar a 1.06 significa ~4x mejor que azar.

**8. Por que no `model.fit(X, Y, batch_size=32)`?**
Con 20 000 ventanas y batch=128 cada epoca hace 156 pasos. Batch=32
daria 625 pasos, mucho mas lento en CPU. Ademas, Adam converge mejor
con batches no muy chicos.

**9. Cuanto tarda el entrenamiento?**
~5 minutos en CPU (4 cores) con 20 000 ventanas y 80 epocas. Sin GPU
en el equipo, TF degrada a CPU sin fallar.

**10. Por que `set_seed(42)` en todo?**
Reproducibilidad: misma semilla -> mismo split estratificado,
misma inicializacion de pesos, mismo barajado. Cualquiera puede
reentrenar y obtener los mismos numeros.


## 15. Decisiones de diseno

Tabla resumen del **por que** de cada eleccion clave. Es la base para
responder preguntas no anticipadas.

| Decision | Valor | Razon |
|---|---|---|
| Arquitectura | SimpleRNN | requisito literal de la actividad |
| Granularidad | char-level | vocab chico, sin tokenizer externo |
| `BLOCK_SIZE` | 32 | cubre 1-2 lineas de codigo |
| `embed_dim` | 48 | compacto, ~94*48 = 4 512 params |
| `hidden` | 64 | balance entre capacidad y velocidad |
| Optimizer | Adam(1e-3) | standard para RNNs pequenas |
| Loss | SparseCategoricalCrossentropy(from_logits) | labels son ints |
| Epochs | 80 | convergente en este dataset |
| Batch size | 128 | standard |
| Sub-muestreo | 20 000 ventanas | CPU-friendly (~5 min) |
| Splits | no hay train/val/test | corpus chico, no hace falta |
| Augmentation | no | el corpus ya es pequeno, no queremos agrandarlo artificialmente |
| Temperatura default | 0.4 | ~70% determinista, mantiene variedad |
| Servidor HTTP | FastAPI (F5) | integraciones externas, pruebas con curl |
| Servidor stdio | JSON-line (F6) | integracion con VSCode (sin HTTP/socket) |
| Extension editor | VSCode (F7) | editor mas usado, extension.js liviana |
| Persistencia | formato .keras | nativo de Keras 3, ~244 KB |


## 16. Limitaciones y extensiones posibles

**Limitaciones conocidas**

- **Memoria corta**: `BLOCK_SIZE=32` significa que el modelo no ve
  contexto mas alla de 32 chars. Una linea larga pierde el inicio.
- **Sin gramatica**: el modelo no valida parentesis ni llaves.
  Sube a 1.0 de loss = todavia ~37% de error por caracter.
- **Sin GPU**: ~5 min de entrenamiento en CPU. Con GPU seria <30 s.
- **150 funciones**: el corpus es chico. La calidad escala con datos.
- **No es ejecutable**: el modelo genera sintaxis C razonable pero
  no se puede correr como codigo. Hay que entenderlo como sugerencia.

**Extensiones naturales**

- **GRU / LSTM**: cambiar `SimpleRNN` por `GRU(64)` o `LSTM(64)`.
  Mejora la memoria de largo plazo pero deja de ser Vanilla.
- **Token-level con BPE**: en vez de caracteres, usar sub-palabras
  con Byte-Pair Encoding. Vocab mas rico, generacion mas coherente.
- **Beam search**: en vez de muestrear un solo char, mantener los
  K candidatos mas probables y elegir el mejor al final.
- **Attention**: agregar una capa de atencion para que el modelo
  pueda mirar mas alla de la ultima `BLOCK_SIZE` posiciones.
- **Export a TFLite**: para correr el modelo en mobile o edge devices.
- **Linter que valide parens**: post-procesar la salida para balancear
  parentesis y llaves (reglas simples).
- **Dataset mas grande**: scrappear GitHub o usar un corpus de
  un curso de sistemas. La perdida bajaria sustancialmente.


## 17. Resumen y ejecucion

**Setup inicial (una sola vez)**

```fish
# El venv unificado vive en la raiz del repo (IA/.venv)
cd ~/IA
source .venv/bin/activate.fish
# Si hubiera que reinstalar dependencias desde cero:
# pip install -r 1-game/requirements.txt
# pip install -r 2-CNN/requirements.txt
# pip install -r 3-RNN/requirements.txt
```

**Pipeline completo (F1 -> F3)**

```fish
# El venv ya esta activo desde la raiz del repo
python src/sample_clean.py                  # F1: 150 funciones en dataset/sample/
python src/preprocess.py                    # F2: X.npy, Y.npy en dataset/processed/
python src/train.py --epochs 80             # F3: rnn_v1.keras en models/
```

**Probar la prediccion (4 formas)**

```fish
# F4: CLI
python src/predict.py --prompt "int sum" --max-new 60 --temperature 0.4

# F5: API HTTP
uvicorn src.api:app --port 8000 &
curl -X POST http://127.0.0.1:8000/predict \\
     -H 'Content-Type: application/json' \\
     -d '{"prompt":"int sum","max_new":40,"temperature":0.4}'

# F6: Servidor stdio (lo que usa la extension)
echo '{"id":1,"method":"complete","prefix":"int sum","max_new":40}' \\
     | python src/server_stdio.py

# F7: Extension VSCode (modo dev)
code vscode-extension/                      # F5 en VSCode, Ctrl+Shift+Space en un .c
```

**Ver este notebook**

```fish
cd ~/IA
./scripts/jupyter.fish
# Se abre en el navegador. Navega a 3-RNN/docs/notebooks/ y abre este archivo.
# En la esquina superior derecha elegi el kernel unificado
# "IA (Python 3.11 + TF)" — tiene TF 2.16.1 + todo lo necesario.
```

**Workflow de demo al profesor (5 min)**

1. **Mostrar el notebook** (en Jupyter): recorre las secciones F1 a F7.
2. **Mostrar el CLI** (terminal): `python src/predict.py --prompt "int sum"`.
3. **Mostrar el API HTTP** (terminal): curl a `/predict`.
4. **Mostrar el stdio** (terminal): echo + pipe.
5. **Mostrar la extension** (VSCode): abrir un .c, Ctrl+Shift+Space.

**Archivos del proyecto**

- Codigo fuente: `3-RNN/src/`
- Configuracion: `3-RNN/requirements.txt`
- Venv unificado: `~/IA/.venv` (raiz del repo, compartido con 1-game y 2-CNN)
- Dataset: `3-RNN/dataset/`
- Modelos entrenados: `3-RNN/models/`
- Extension: `3-RNN/vscode-extension/`
- Documentacion: `3-RNN/docs/`
- Notebooks: `3-RNN/notebooks/entrenamiento.ipynb` (formal) y este (guia)

**Aclaracion final**

Este notebook **no predice codigo C**: es la guia explicativa para
mostrarle al profesor como funciona el pipeline. La prediccion real
ocurre en la extension VSCode (F7), en `predict.py` (F4), en la API
HTTP (F5), o en el servidor stdio (F6).
